# EquityAgent: Data and NLP starter

GitHub repository: <repo-url>

This notebook covers the **Data lead** and **NLP lead** starting tasks:

1. Setup
2. Data check: each tool returns data (prices, technicals, macro, news,
   10-K sections)
3. Kaggle news EDA: label balance, duplicates, headline length, held-out
   sample
4. Classifier evaluation: keyword baseline vs LLM on the held-out sample
5. News prompt chain: output of each step on a live ticker

## 1. Setup

In [ ]:
# --- Colab setup -----------------------------------------------------
import os
import sys

from google.colab import drive, userdata

drive.mount("/content/drive")
REPO = "/content/drive/MyDrive/equity-agent"
assert os.path.isdir(REPO), f"Repo not found at {REPO}; check the folder name"
%cd {REPO}
sys.path.insert(0, REPO)

!pip install -q -r requirements.txt

# Load keys from Colab Secrets before importing src.config
SECRET_KEYS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "FRED_API_KEY",
    "NEWSAPI_KEY",
    "EDGAR_USER_AGENT",
]
loaded = []
for key in SECRET_KEYS:
    try:
        os.environ[key] = userdata.get(key)
        loaded.append(key)
    except Exception:
        print(f"Missing secret: {key}")
print("Loaded:", loaded)

!git config --global user.name "nicole-rowley-msaai"
!git config --global user.email "nicolerowley@sandiego.edu"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import TICKERS
from src.tools import safe_call

## 2. Data check: does every tool return data?

Each call goes through `safe_call`, so a failed API shows as `ok=False`
instead of stopping the notebook.

In [ ]:
from src.tools.filings import extract_sections, get_latest_10k
from src.tools.macro import get_macro_snapshot
from src.tools.market import compute_technicals, get_price_history
from src.tools.news import get_news


def status(result):
    """Every ToolResult field except the data itself."""
    return {k: v for k, v in vars(result).items() if k != "data"}


checks = []
spy = safe_call("prices", get_price_history, "SPY")
for ticker in TICKERS:
    prices = safe_call("prices", get_price_history, ticker)
    checks.append({"ticker": ticker, **status(prices)})
    if prices.ok and spy.ok:
        print(ticker, compute_technicals(prices.data, spy.data))

macro = safe_call("fred", get_macro_snapshot)
checks.append({"ticker": "-", **status(macro)})
pd.DataFrame(checks)

In [ ]:
macro.data if macro.ok else macro.error

In [ ]:
news = safe_call("news", get_news, "NVDA", "Nvidia")
print(news.ok, news.meta, news.error)
pd.DataFrame(news.data[:5]) if news.ok else None

**10-K sections.** Three filer quirks surfaced during iteration, and
`src/tools/filings.py` handles each:

- **JPMorgan** embeds its whole annual report in the 10-K file, so "Item 7"
  is only a cross-reference. The extractor falls back to the plain heading
  "Management's discussion and analysis" when the Item 7 span is too short.
- **Exxon** reorganized into ExxonMobil Holdings Corp in mid-2026. The
  ticker now maps to a new SEC registrant with no 10-K, so the loader
  retries the prior entity's CIK from `config.PREDECESSOR_CIKS`.
- **Heavy filers** can push a 10-K off the first page of the SEC
  submissions feed, so the loader follows the paginated files.

The `cik` column below shows which registrant each 10-K came from.

In [ ]:
secs = None
for ticker in TICKERS:
    filing = safe_call("edgar", get_latest_10k, ticker)
    if not filing.ok:
        print(ticker, filing.error)
        continue
    secs = extract_sections(filing.data)
    words = {k: len(v.split()) for k, v in secs.items()}
    print(ticker, "cik", filing.data["cik"], filing.data["filed"], words)

if secs:
    print()
    print(secs["mdna"][:500])

## 3. Kaggle news EDA

The labeled dataset is
[Kaggle: Sentiment Analysis for Financial News](https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news)
(the Financial PhraseBank). The cell below downloads it once with the Kaggle
API; add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab Secrets, or drop
`all-data.csv` into `data/kaggle/` by hand.

In [ ]:
# Download the Kaggle dataset once; it stays in Drive afterward.
from src.config import KAGGLE_CSV

DATASET = "ankurzing/sentiment-analysis-for-financial-news"

if not KAGGLE_CSV.exists():
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !kaggle datasets download -d {DATASET} -p data/kaggle --unzip
print("Kaggle CSV present:", KAGGLE_CSV.exists())

In [ ]:
from src.chains.kaggle import (
    LABELS, clean_kaggle, held_out_sample, load_kaggle,
)

raw = load_kaggle()
print("Rows:", len(raw), "| Columns:", list(raw.columns))
print("Exact duplicate headlines:", raw.duplicated("headline").sum())
print("Missing values:", raw.isna().sum().to_dict())
df = clean_kaggle(raw)
print("Rows after cleaning:", len(df))

In [ ]:
counts = df["sentiment"].value_counts().reindex(LABELS)
print((counts / counts.sum()).round(3))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.index, counts.values, color="#3b6fb6", width=0.6)
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
ax.set_title("Label balance: mostly neutral")
ax.set_ylabel("Headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
df["words"] = df["headline"].str.split().str.len()
print(df.groupby("sentiment")["words"].describe().round(1))

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(df["words"], bins=30, color="#3b6fb6")
ax.set_title("Headline length (words)")
ax.set_xlabel("Words")
ax.set_ylabel("Headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
test = held_out_sample(df, n=300)  # fixed seed, stratified
test.to_csv("data/kaggle/test_sample.csv", index=False)
test["sentiment"].value_counts()

## 4. Classifier evaluation: keyword baseline vs LLM

The LLM classifier is scored on the fixed 300-headline held-out sample with
a sentiment-only prompt. The live news chain uses a longer prompt that also
asks for relevance, but the Kaggle headlines have no target company, so that
instruction would only add noise here. Because LLM output varies between
runs (about 2 points of accuracy at n = 300), the score is the mean of
three runs.

In [ ]:
from src.chains.kaggle import keyword_baseline, score, to_articles
from src.chains.news_chain import CLASSIFY_SENTIMENT_SYSTEM, classify

METRICS = ("accuracy", "macro_f1")
N_RUNS = 3  # LLM output varies run to run; report the mean, not one draw

base = score(test["sentiment"], test["headline"].map(keyword_baseline))
print("Keyword baseline:", {k: base[k] for k in METRICS})

articles = to_articles(test)
runs = []
for r in range(N_RUNS):
    labels = classify(articles, system=CLASSIFY_SENTIMENT_SYSTEM)
    pred = [
        labels.get(i, {}).get("sentiment", "neutral")
        for i in range(len(test))
    ]
    runs.append(score(test["sentiment"], pred))
    print(f"LLM run {r + 1}:", {k: runs[-1][k] for k in METRICS})

mean = {k: round(sum(s[k] for s in runs) / N_RUNS, 3) for k in METRICS}
print("LLM mean of", N_RUNS, "runs:", mean)

llm = runs[-1]  # last run's per-class report and confusion matrix
print(llm["report"])

In [ ]:
cm = llm["confusion"]

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), LABELS)
ax.set_yticks(range(3), LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(3):
    for j in range(3):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, cm[i, j], ha="center", va="center", color=color)
ax.set_title("LLM classifier confusion matrix")
plt.show()

## 5. News prompt chain: every step printed

**Iteration:** the first run ingested 99 articles, and the digest cited
market roundups that only mentioned Nvidia in passing. Two changes fixed
this: the NewsAPI query now requires the company name in the headline
(`qInTitle`), and the classify step labels each item's relevance so
roundups are dropped before extraction. The trace below shows the new
`RELEVANCE_FILTER` step between `CLASSIFY` and `EXTRACT`.

In [ ]:
from src.chains.news_chain import ingest, preprocess, run_news_chain

trace = []
digest = run_news_chain("NVDA", "Nvidia", trace=trace)
for step in trace:
    print(f"\n=== {step['step'].upper()} ===")
    out = step["output"]
    is_item_map = isinstance(out, dict) and all(
        isinstance(k, int) for k in out
    )
    if is_item_map and len(out) > 5:
        out = dict(list(out.items())[:5])  # show first 5 items
    print(out)

In [ ]:
# What the relevance filter removed (cached calls, so no extra API cost).
steps = {s["step"]: s["output"] for s in trace}
chain_labels = steps["classify"]
clean = preprocess(ingest("NVDA", "Nvidia"), "Nvidia", "NVDA")

dropped = [
    a.title for a in clean
    if chain_labels.get(a.id, {}).get("relevance") == "low"
]
print(f"Dropped {len(dropped)} low-relevance articles. Examples:")
for title in dropped[:5]:
    print(" -", title)

In [ ]:
print(digest.model_dump_json(indent=2))